## FROM README
### AnnData Structure
The data are released as an AnnData files and contains gene expression information for 36,601 features/genes contained with the 10x Genomics 2020-A human reference. Alignments for singleome RNAseq (10x v3.1 3') and multiome (10x) were performed identically with cellranger (v6.1.2) and cellranger-arc (v2.0.0), respectively.

"final-nuclei" objects have had low quality nuclei removed, while "all-nuclei" objects contain all nuclei identified by cellranger.

The AnnData files within each folder all follow the same structure:

AnnData.X - Contains log-normalized unique molecular identifier counts per 10,000

AnnData.layers = Contains raw counts of unique molecular identifiers ("UMIs")

AnnData.obs - Contains per-cell donor-, library-, and cell-level metadata. Some notable features:
* library_prep: Identifier for the specific library preparation batch or protocol used to process the cell's RNA, which converts RNA molecules into a barcoded, sequencer-ready DNA library - string
* Donor ID: ID of each donor - string
* Method: Method to isolate cells - "3' 10x v3.1" or "3' 10x Multiome"
* Sex: Biological sex, defined by presence of Y chromosome - "Male" or "Female"
* Age at Death: Donor's age at death - int
* Race (choice=White) - bool
* Race (choice=Black/ African American) - bool
* Race (choice=Asian) - bool
* Race (choice=American Indian/ Alaska Native) - bool
* Race (choice=Native Hawaiian or Pacific Islander) - bool
* Race (choice=Unknown or unreported) - bool
* Race (choice=Other) - bool
* Hispanic/Latino - bool
* Years of education - int
* PMI: Postmortem interval in hours - float
* APOE Genotype: Apolipoprotein E (APOE) alleles for a donor (Protective APOE2 = 2, Reference APOE3 = 3, Risk associated APOE4 = 4) - "2/2", "2/3", "3/3", "3/4", "4/4"
* Thal: Ordinal extent of the anatomical distribution of amyloid beta plaque deposits - "Thal 0", "Thal 1", "Thal 2", "Thal 3", "Thal 4", or "Thal 5"
* Braak: Ordinal extent of the anatomical distribution of neurofibrillary tangles (NFTs) - "Braak 0", "Braak I", "Braak II", "Braak III", "Braak IV", "Braak V", or "Braak VI"
* CERAD score: Ordinal semiquantitative neuritic plaque density - "Absent", "Sparse", "Moderate", or "Frequent"
* Overall AD neuropathological Change: Ordinal Overall Alzheimer's disease neuropathologic change score - "Not AD", "Low", "Intermediate", or "High"
* LATE: Ordinal extent of Limbic-predominant Age-related TDP-43 Encephalopathy-Neuropathologic Change - "Unclassifiable", "Not Identified", "LATE Stage 1", "LATE Stage 2", or "LATE Stage 3"
* Highest Lewy Body Disease: Ordinal Anatomical distribution of Lewy bodies - "Not Identified (olfactory bulb not assessed)", "Not Identified (olfactory bulb assessed)", "Not Identified (olfactory bulb assessed)", "Olfactory bulb only", "Amygdala-predominant", "Brainstem-predominant", "Limbic (Transitional)", or "Neocortical (Diffuse)".
* Cognitive Status: Binary clinical determination on if someone has dementia or not - "No dementia" or "Dementia"
* Class: Major cell type class (e.g. Non-neuronal and non-neural)
* Subclass: Cell type subclass (e.g. Microglia-PVM; which includes Microglia and other immune cells) 
* Supertype: The specific, mappable cell type (e.g. Micro-PVM_2 or Lymphocyte)

AnnData.obsm - Contains the scVI latent representation computed on the nuclei ("X_scVI") and umap coordinates from the nearest neighbor graph computed on this space ("X_umap"). Only available in "final-nuclei" tagged objects.

AnnData.var - Contains gene names and ensembl IDs for each feature

AnnData.uns - Contains official colors from our cell type taxonomy in "Subclass_colors" and "Supertype_colors"

In [2]:
# Data handling
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt

import os
import warnings
warnings.filterwarnings("ignore")

In [4]:
metadata_full = pd.read_csv("/data/shared/alzgene26/data/seaAD/SEAAD_DFC_RNAseq_final-nuclei_metadata.2026-06-22.csv")

In [5]:
col_list_full = metadata_full.columns.tolist()
print(f"columns: {col_list_full} \nwith length: {len(col_list_full)}")

columns: ['exp_component_name', 'sample_id', 'Neurotypical reference', 'Donor ID', 'Organism', 'Brain Region', 'Sex', 'Gender', 'Age at Death', 'Race (choice=White)', 'Race (choice=Black/ African American)', 'Race (choice=Asian)', 'Race (choice=American Indian/ Alaska Native)', 'Race (choice=Native Hawaiian or Pacific Islander)', 'Race (choice=Unknown or unreported)', 'Race (choice=Other)', 'specify other race', 'Hispanic/Latino', 'Highest level of education', 'Years of education', 'PMI', 'Fresh Brain Weight', 'Brain pH', 'Overall AD neuropathological Change', 'Thal', 'Braak', 'CERAD score', 'Overall CAA Score', 'Highest Lewy Body Disease', 'Total Microinfarcts (not observed grossly)', 'Total microinfarcts in screening sections', 'Atherosclerosis', 'Arteriolosclerosis', 'LATE', 'Cognitive Status', 'Last CASI Score', 'Interval from last CASI in months', 'Last MMSE Score', 'Interval from last MMSE in months', 'Last MOCA Score', 'Interval from last MOCA in months', 'APOE Genotype', 'Prima

In [6]:
metadata_full[:5]

,exp_component_name,sample_id,Neurotypical reference,Donor ID,Organism,Brain Region,Sex,Gender,Age at Death,Race (choice=White),...,Number of reads,Number of UMIs,Genes detected,Doublet score,Fraction mitochondrial UMIs,Severely Affected Donor,Class,Subclass,Supertype,Used in analysis
0,AAACCCAAGCGTCTGC-L8TX_210805_01_B03-1153814324,AAACCCAAGCGTCTGC-L8TX_210805_01_B03-1153814324,False,H20.33.016,human,DFC,Female,Female,93.0,Checked,...,90615.0,34882.0,7246,0.072289,0.000602,N,Neuronal: Glutamatergic,L2/3 IT,L2/3 IT_13,True
1,AAACCCAAGTGGTTAA-L8TX_210805_01_B03-1153814324,AAACCCAAGTGGTTAA-L8TX_210805_01_B03-1153814324,False,H20.33.016,human,DFC,Female,Female,93.0,Checked,...,22026.0,7869.0,3237,0.000000,0.001271,N,Neuronal: GABAergic,Sst,Sst_19,True
2,AAACCCACATCCGAAT-L8TX_210805_01_B03-1153814324,AAACCCACATCCGAAT-L8TX_210805_01_B03-1153814324,False,H20.33.016,human,DFC,Female,Female,93.0,Checked,...,154515.0,34773.0,7263,0.060241,0.000690,N,Neuronal: Glutamatergic,L6 CT,L6 CT_2,True
3,AAACCCACATCTAACG-L8TX_210805_01_B03-1153814324,AAACCCACATCTAACG-L8TX_210805_01_B03-1153814324,False,H20.33.016,human,DFC,Female,Female,93.0,Checked,...,56441.0,12069.0,4914,0.084337,0.000580,N,Non-neuronal and Non-neural,Astrocyte,Astro_2,True
4,AAACCCAGTATTTCTC-L8TX_210805_01_B03-1153814324,AAACCCAGTATTTCTC-L8TX_210805_01_B03-1153814324,False,H20.33.016,human,DFC,Female,Female,93.0,Checked,...,129551.0,38153.0,7491,0.048193,0.000970,N,Neuronal: Glutamatergic,L2/3 IT,L2/3 IT_1,True


In [7]:
# Some metadata information
print("Braak stage: ", metadata_full["Braak"].unique())
print("CERAD score: ", metadata_full["CERAD score"].unique())
print("Cognitive status: ", metadata_full["Cognitive Status"].unique())
print("Overall AD neuropathological Change: ", metadata_full["Overall AD neuropathological Change"].unique())

Braak stage:  ['Braak IV' 'Braak VI' 'Braak V' 'Braak III' 'Braak 0' 'Braak II']
CERAD score:  ['Moderate' 'Frequent' 'Absent' 'Sparse']
Cognitive status:  ['Dementia' 'No dementia']
Overall AD neuropathological Change:  ['Intermediate' 'High' 'Low' 'Not AD']


In [8]:
# Cleaning of metadata, drop majority of columns
metadata = metadata_full.drop(metadata_full.loc[:,'Last CASI Score':'Fraction mitochondrial UMIs'].columns, axis=1)
metadata = metadata.drop(metadata.loc[:,'Race (choice=White)':'Years of education'].columns, axis=1)

# Remove columns with no extra information (ex. Brain Region has only DFC)
metadata = metadata.drop(columns=["Neurotypical reference", "Organism", "Brain Region", "Used in analysis"])

In [9]:
# Remaining columns
col_list = metadata.columns.tolist()
print(col_list)

['exp_component_name', 'sample_id', 'Donor ID', 'Sex', 'Gender', 'Age at Death', 'PMI', 'Fresh Brain Weight', 'Brain pH', 'Overall AD neuropathological Change', 'Thal', 'Braak', 'CERAD score', 'Overall CAA Score', 'Highest Lewy Body Disease', 'Total Microinfarcts (not observed grossly)', 'Total microinfarcts in screening sections', 'Atherosclerosis', 'Arteriolosclerosis', 'LATE', 'Cognitive Status', 'Severely Affected Donor', 'Class', 'Subclass', 'Supertype']


In [10]:
metadata[:5]

,exp_component_name,sample_id,Donor ID,Sex,Gender,Age at Death,PMI,Fresh Brain Weight,Brain pH,Overall AD neuropathological Change,...,Total Microinfarcts (not observed grossly),Total microinfarcts in screening sections,Atherosclerosis,Arteriolosclerosis,LATE,Cognitive Status,Severely Affected Donor,Class,Subclass,Supertype
0,AAACCCAAGCGTCTGC-L8TX_210805_01_B03-1153814324,AAACCCAAGCGTCTGC-L8TX_210805_01_B03-1153814324,H20.33.016,Female,Female,93.0,7.733333,1242.0,6.8,Intermediate,...,0.0,0.0,Moderate,Severe,Not Identified,Dementia,N,Neuronal: Glutamatergic,L2/3 IT,L2/3 IT_13
1,AAACCCAAGTGGTTAA-L8TX_210805_01_B03-1153814324,AAACCCAAGTGGTTAA-L8TX_210805_01_B03-1153814324,H20.33.016,Female,Female,93.0,7.733333,1242.0,6.8,Intermediate,...,0.0,0.0,Moderate,Severe,Not Identified,Dementia,N,Neuronal: GABAergic,Sst,Sst_19
2,AAACCCACATCCGAAT-L8TX_210805_01_B03-1153814324,AAACCCACATCCGAAT-L8TX_210805_01_B03-1153814324,H20.33.016,Female,Female,93.0,7.733333,1242.0,6.8,Intermediate,...,0.0,0.0,Moderate,Severe,Not Identified,Dementia,N,Neuronal: Glutamatergic,L6 CT,L6 CT_2
3,AAACCCACATCTAACG-L8TX_210805_01_B03-1153814324,AAACCCACATCTAACG-L8TX_210805_01_B03-1153814324,H20.33.016,Female,Female,93.0,7.733333,1242.0,6.8,Intermediate,...,0.0,0.0,Moderate,Severe,Not Identified,Dementia,N,Non-neuronal and Non-neural,Astrocyte,Astro_2
4,AAACCCAGTATTTCTC-L8TX_210805_01_B03-1153814324,AAACCCAGTATTTCTC-L8TX_210805_01_B03-1153814324,H20.33.016,Female,Female,93.0,7.733333,1242.0,6.8,Intermediate,...,0.0,0.0,Moderate,Severe,Not Identified,Dementia,N,Neuronal: Glutamatergic,L2/3 IT,L2/3 IT_1


In [12]:
metadata["sample_id"].unique()

array(['AAACCCAAGCGTCTGC-L8TX_210805_01_B03-1153814324',
       'AAACCCAAGTGGTTAA-L8TX_210805_01_B03-1153814324',
       'AAACCCACATCCGAAT-L8TX_210805_01_B03-1153814324', ...,
       'TTTGTTGGTCTCACTG-L8XR_220804_01_C03-9019aaa5dac7af6adde1395be0462cbe50fe03c6',
       'TTTGTTGGTTGGTTAG-L8XR_220804_01_C03-9019aaa5dac7af6adde1395be0462cbe50fe03c6',
       'TTTGTTGGTTTATGGG-L8XR_220804_01_C03-9019aaa5dac7af6adde1395be0462cbe50fe03c6'],
      shape=(1357145,), dtype=object)

In [21]:
donors = list(metadata["Donor ID"].unique())
print(f"nr of donors: {len(donors)}")
print(f"donor IDs: {donors}")

nr of donors: 80
donor IDs: ['H20.33.016', 'H21.33.002', 'H21.33.036', 'H20.33.012', 'H21.33.012', 'H20.33.039', 'H20.33.020', 'H20.33.001', 'H20.33.041', 'H20.33.024', 'H21.33.022', 'H21.33.032', 'H21.33.029', 'H21.33.017', 'H20.33.032', 'H21.33.019', 'H20.33.034', 'H21.33.008', 'H20.33.036', 'H21.33.046', 'H20.33.030', 'H21.33.047', 'H20.33.027', 'H20.33.031', 'H20.33.018', 'H20.33.019', 'H19.33.004', 'H20.33.028', 'H21.33.028', 'H20.33.017', 'H21.33.043', 'H20.33.046', 'H21.33.007', 'H21.33.016', 'H20.33.037', 'H21.33.003', 'H21.33.042', 'H20.33.029', 'H20.33.013', 'H20.33.040', 'H21.33.027', 'H20.33.044', 'H21.33.015', 'H20.33.045', 'H20.33.014', 'H21.33.006', 'H20.33.008', 'H21.33.031', 'H21.33.011', 'H21.33.001', 'H21.33.044', 'H20.33.015', 'H21.33.023', 'H20.33.004', 'H21.33.004', 'H21.33.037', 'H20.33.025', 'H20.33.033', 'H20.33.038', 'H20.33.035', 'H21.33.018', 'H21.33.033', 'H20.33.011', 'H20.33.005', 'H21.33.020', 'H21.33.030', 'H21.33.045', 'H21.33.026', 'H21.33.013', 'H21.

In [1]:
# Inspection of .h5ad file
import h5py
file = h5py.File("data/seaAD/SEAAD_DFC_RNAseq_final-nuclei.2026-06-22.h5ad", "r")
print(list(file.keys()))

['X', 'layers', 'obs', 'obsm', 'obsp', 'uns', 'var', 'varm', 'varp']


In [2]:
print(file["X"])

<HDF5 group "/X" (3 members)>


In [ ]:
print(file["X"]["data"].shape)
print(dict(file["X"].attrs))
print(len(file["obs"]))
print(len(file["var"]))

(7748147192,)
{'encoding-type': 'csr_matrix', 'encoding-version': '0.1.0', 'shape': array([1357145,   36601])}
131
4


In [6]:
print(list(file["obs"].keys()))
print(list(file["var"].keys()))
print(list(file["layers"].keys()))

['APOE Genotype', 'ATAC_Confidently_mapped_read_pairs', 'ATAC_Fraction_of_genome_in_peaks', 'ATAC_Fraction_of_high_quality_fragments_in_cells', 'ATAC_Fraction_of_high_quality_fragments_overlapping_TSS', 'ATAC_Fraction_of_high_quality_fragments_overlapping_peaks', 'ATAC_Fraction_of_transposition_events_in_peaks_in_cells', 'ATAC_Mean_raw_read_pairs_per_cell', 'ATAC_Median_high_quality_fragments_per_cell', 'ATAC_Non-nuclear_read_pairs', 'ATAC_Number_of_peaks', 'ATAC_Percent_duplicates', 'ATAC_Q30_bases_in_barcode', 'ATAC_Q30_bases_in_read_1', 'ATAC_Q30_bases_in_read_2', 'ATAC_Q30_bases_in_sample_index_i1', 'ATAC_Sequenced_read_pairs', 'ATAC_TSS_enrichment_score', 'ATAC_Unmapped_read_pairs', 'ATAC_Valid_barcodes', 'Age at Death', 'Arteriolosclerosis', 'Atherosclerosis', 'Braak', 'Brain Region', 'Brain pH', 'CERAD score', 'Class', 'Cognitive Status', 'Donor ID', 'Doublet score', 'Fraction mitochondrial UMIs', 'Fresh Brain Weight', 'GEX_Estimated_number_of_cells', 'GEX_Fraction_of_transcript

In [ ]:
data_path = "data/seaAD/SEAAD_DFC_RNAseq_final-nuclei.2026-06-22.h5ad"
# ensure that the file was properly saved

adata_full = sc.read_h5ad(data_path, backed=True)
print(f"Successfully read and saved {adata_full.n_obs} cells")

In [ ]:
# Show information about data file
print(adata_full)

In [ ]:
# Show example
print(adata_full.obs.head())

In [ ]:
# Print all available category names you can use for plotting
print(adata_full.obs.columns)

In [ ]:
# Sparsity Check
print(f"Sparse AnnData: {adata_full.n_obs} n_obs x {adata_full.n_vars} n_vars")
sparsity = 1 - adata_full.X.nnz / (adata_full.n_obs * adata_full.n_vars)
print(f"Sparsity: {sparsity:.2%}")